In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch09-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "decision-trees.py"

# 沿著 MACCS 位元讀一棵分類樹

Colab 請選 2026.07，第一次載入實驗資料需要連網。下格寫入暫存程式，最後一格執行並顯示圖。做小練習時，把程式接在分析格下方，重新執行兩格。

致突變性資料取自 Lou 等人（2023）的 [Maom Lab 鏡像](https://huggingface.co/datasets/maomlab/MutagenLou2023)（CC BY 4.0）。沿用鏡像測試檔，清理結構後從開發資料分出驗證集。

In [ ]:
%%writefile "{script}"
# MutagenLou2023 (Maom Lab mirror, CC BY 4.0; Lou et al., 2023).
# Original rows/labels preserved in the cached files. Only structure enters X.
from pathlib import Path
from urllib.request import urlopen
import csv
import hashlib
import io
import os
import tempfile
import numpy as np
from rdkit import Chem
from sklearn.model_selection import train_test_split

def mutagen_splits():
    revision = "a9780a60774eb522b78bd92b83c30d8deae00ff5"
    source = f"https://huggingface.co/datasets/maomlab/MutagenLou2023/resolve/{revision}/train_test"
    base = Path(os.environ.get("DEEPCHEM_DATA_DIR", tempfile.gettempdir())) / "mutagenlou-a9780a6"
    base.mkdir(parents=True, exist_ok=True)
    expected = {"train": "ad0d4e2850afdc07c130c3679cc8ae2ad9f7c80999158325d8de986c39f950e1",
                "test": "08786199fbbdb6db614d5f65a5abdfcd26d882f0cc844bbeb4ccfa241127b394"}
    groups = {}
    for name, digest in expected.items():
        file = base / (name + ".csv")
        raw = file.read_bytes() if file.exists() else urlopen(f"{source}/{name}.csv", timeout=90).read()
        if hashlib.sha256(raw).hexdigest() != digest:
            raise ValueError(f"MutagenLou2023 checksum mismatch: {name}")
        if not file.exists():
            file.write_bytes(raw)
        labels = {}
        for row in csv.DictReader(io.StringIO(raw.decode("utf-8-sig"))):
            mol = Chem.MolFromSmiles(row["SMILES"])
            if mol is not None:
                y = int(row["Y"])
                assert y in (0, 1)
                labels.setdefault(Chem.MolToSmiles(mol), set()).add(y)
        groups[name] = labels
    # Preserve mirrored test membership: no test structure may enter development.
    dev = [(s, next(iter(v))) for s,v in groups["train"].items() if len(v)==1 and s not in groups["test"]]
    test = [(s, next(iter(v))) for s,v in groups["test"].items() if len(v)==1]
    fit, valid = train_test_split(dev, test_size=0.15, random_state=42, stratify=[y for _,y in dev])
    return [(np.array([s for s,_ in rows]), np.array([y for _,y in rows])) for rows in (fit,valid,test)]

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem

SEED = 42
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)



def make_dataset(smiles, y, featurize):
    X = np.stack([featurize(Chem.MolFromSmiles(s)) for s in smiles])
    return dc.data.NumpyDataset(X=X, y=y, ids=smiles)

from rdkit.Chem import MACCSkeys
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text

def maccs(mol):
    return np.asarray(MACCSkeys.GenMACCSKeys(mol), dtype=np.uint8)

train, valid, test = [make_dataset(s, y, maccs) for s, y in mutagen_splits()]
depths = [1, 2, 4, 8, None]
rows = []
models = []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    tree.fit(train.X, train.y.ravel())
    rows.append((tree.score(train.X, train.y.ravel()), tree.score(valid.X, valid.y.ravel())))
    models.append(tree)
best = np.argmax([v for t, v in rows])
print("depth train valid")
for depth, (t, v) in zip(depths, rows):
    print(depth, f"{t:.3f}", f"{v:.3f}")
print("selected", depths[best], "test accuracy", f"{models[best].score(test.X, test.y.ravel()):.3f}")
short = models[1]
names = [f"bit {i}" for i in range(train.X.shape[1])]
print(export_text(short, feature_names=names))
print("node counts (negative, positive)", np.rint(short.tree_.value[:, 0] * short.tree_.n_node_samples[:, None]).astype(int))
fig, ax = plt.subplots(figsize=(9.2, 4.6), layout="constrained")
plot_tree(short, ax=ax, feature_names=names, class_names=["negative", "positive"],
          filled=True, rounded=True, fontsize=8, precision=2)
fig.savefig(figure_dir / "09-06-decision-trees.png", dpi=150)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "09-06-decision-trees.png")))